# Supplementary Materials

Data and code required to reproduce additional supplementary materials data not already included in the other notebooks.

In [1]:
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import scipy.stats as sp

from utils.utils import *
from utils.variables import *

## Read in data

In [2]:
df_pids, df_ctrs, df_general, df_comparisons, df_users = retrieve_analysed_data('all')

Removing 7 transcripts that are not appropriate for CBT
N = 227


condition        model 
cognitive_layer  claude    25
                 gemini    26
                 gpt4      26
                 llama     24
human_therapist  human     26
standalone_llms  claude    25
                 gemini    24
                 gpt4      27
                 llama     24
dtype: int64

## Session duration & latency

Supplementary Table 3

In [3]:
print('Latencies')
print(df_pids.groupby(['condition'])['mean_response_latency_seconds'].agg(['mean','std']).round(2))
print(' ')
for combo in [['cognitive_layer','standalone_llms'],['cognitive_layer','human_therapist']]:
    vec_A = df_pids.loc[(df_pids['condition']==combo[0]),'mean_response_latency_seconds']
    vec_B = df_pids.loc[(df_pids['condition']==combo[1]),'mean_response_latency_seconds']
    stat = sp.ttest_ind(vec_A,vec_B)
    print(f"{combo[0]} vs {combo[1]}: t({stat.df:.0f}) = {stat.statistic:.3f}, p = {readable_pvalue(stat.pvalue)}")

Latencies
                  mean    std
condition                    
cognitive_layer   5.43   2.14
human_therapist  83.34  31.24
standalone_llms   2.93   1.38
 
cognitive_layer vs standalone_llms: t(199) = 9.855, p = 6.50e-19***
cognitive_layer vs human_therapist: t(125) = -25.119, p = 1.10e-50***


In [4]:
df = pd.DataFrame(
    columns=['latency','duration'],
    index=['ctrs','waisr','mood','feel_human','belief_human']
    )

# ctrs
vec_A = df_ctrs.groupby('pid')['human1_score'].mean().reset_index()
vec_B = df_pids.groupby('pid')[['mean_response_latency_seconds','session_duration_minutes']].mean().reset_index()
combined = pd.merge(vec_A,vec_B,on='pid',how='outer')

stat = sp.spearmanr(combined['human1_score'],combined['mean_response_latency_seconds'])
stars = ''.join([c for c in readable_pvalue(stat.pvalue) if c == '*'])
df.loc['ctrs','latency'] = f"{stat.correlation:.2f}{stars}"

stat = sp.spearmanr(combined['human1_score'],combined['session_duration_minutes'])
stars = ''.join([c for c in readable_pvalue(stat.pvalue) if c == '*'])
df.loc['ctrs','duration'] = f"{stat.correlation:.2f}{stars}"

# waisr
waisr_df = df_users.copy().merge(df_pids[['pid','condition','model']],on='pid',how='left')

cols = [x for x in waisr_df.columns if 'waisr_' in x]
for col in cols:
    waisr_df[col] = waisr_df[col].str.lower().replace({
        'strong disagree': 1,
        'disagree': 2,
        'neutral': 3,
        'agree': 4,
        'strong agree': 5,   
    }).astype(int)

waisr_df['waisr_overall'] = waisr_df[cols].mean(axis=1)

combined = (
    waisr_df[['pid','waisr_overall']]
    .merge(df_pids[['pid','mean_response_latency_seconds','session_duration_minutes']],on='pid',how='outer')
)

stat = sp.spearmanr(combined['waisr_overall'],combined['mean_response_latency_seconds'])
stars = ''.join([c for c in readable_pvalue(stat.pvalue) if c == '*'])
df.loc['waisr','latency'] = f"{stat.correlation:.2f}{stars}"

stat = sp.spearmanr(combined['waisr_overall'],combined['session_duration_minutes'])
stars = ''.join([c for c in readable_pvalue(stat.pvalue) if c == '*'])
df.loc['waisr','duration'] = f"{stat.correlation:.2f}{stars}"

# mood
df_mood = df_users[['pid','mood_t0','mood_t1']].copy()
df_mood['mood_change'] = df_mood['mood_t1'] - df_mood['mood_t0']

combined = df_mood.merge(df_pids[['pid','mean_response_latency_seconds','session_duration_minutes']],on='pid',how='outer')

stat = sp.spearmanr(combined['mood_change'],combined['mean_response_latency_seconds'])
stars = ''.join([c for c in readable_pvalue(stat.pvalue) if c == '*'])
df.loc['mood','latency'] = f"{stat.correlation:.2f}{stars}"

stat = sp.spearmanr(combined['mood_change'],combined['session_duration_minutes'])
stars = ''.join([c for c in readable_pvalue(stat.pvalue) if c == '*'])
df.loc['mood','duration'] = f"{stat.correlation:.2f}{stars}"

# felt/believed human
df_humanness = df_users[['pid','feel_human','belief_human']].copy()

combined = df_humanness.merge(df_pids[['pid','mean_response_latency_seconds','session_duration_minutes']],on='pid',how='outer')

for outcome in ['feel_human','belief_human']:
    stat = sp.spearmanr(combined[outcome],combined['mean_response_latency_seconds'])
    stars = ''.join([c for c in readable_pvalue(stat.pvalue) if c == '*'])
    df.loc[outcome,'latency'] = f"{stat.correlation:.2f}{stars}"

    stat = sp.spearmanr(combined[outcome],combined['session_duration_minutes'])
    stars = ''.join([c for c in readable_pvalue(stat.pvalue) if c == '*'])
    df.loc[outcome,'duration'] = f"{stat.correlation:.2f}{stars}"

# Display
display(df)

,latency,duration
ctrs,0.18**,0.12
waisr,0.19**,0.25***
mood,0.13,0.12
feel_human,0.16*,0.18**
belief_human,0.13,0.21**


## Mood change

As a user experience measure (not a clinical outcome)

### Initial analysis

In [10]:
df = df_users[['pid','mood_t0','mood_t1','feel_human','belief_human']].copy()
df['mood_change'] = df['mood_t1'] - df['mood_t0']

df = df.merge(df_pids[['pid','condition','model']],on='pid',how='left')

display(df.groupby('condition')['mood_change'].agg(['mean','sem']))

# ------------------------------------------------------------------------------------------------------------------------------------
# Run models
# ------------------------------------------------------------------------------------------------------------------------------------

# --- 2 x 4 model
model_df = df.loc[df['condition']!='human_therapist',].copy().reset_index(drop=True)

model_fit = smf.ols(
    f'mood_change ~ mood_t0 + C(model) * C(condition) * belief_human + C(model) * C(condition) * feel_human',
    data=model_df).fit()

display_residuals(model_fit)
anova_table = format_anova_table(model_fit)

print('[2 x 4 model]')
print(anova_table)

# --- 3-way model
model_df = df.groupby(['pid','condition'])[[f'mood_change','mood_t0','belief_human','feel_human']].mean().reset_index()

model_fit = smf.ols(
    f'mood_change ~ mood_t0 + C(condition)*belief_human + C(condition)*feel_human',
    data=model_df).fit()

display_residuals(model_fit)
anova_table = format_anova_table(model_fit)

print('[3-way model]')
print(anova_table)

print(' ')
print(anova_pairwise_comparisons(model_fit))

# ------------------------------------------------------------------------------------------------------------------------------------
# Plot
# ------------------------------------------------------------------------------------------------------------------------------------
df['mood_change_percent'] = df['mood_change'] / 6

summary = df.groupby(['condition'])['mood_change'].agg(['mean','sem']).reset_index()
fig = px.bar(
    summary,
    x='condition',
    y='mean',
    error_y='sem',
    color='condition',
    title='Mood change',
    barmode='stack',
    width=300,
    height=400,
    labels={'mean': 'Improvement'},
    category_orders={
        'condition': CONDITION_ORDER,
        'model': ['claude','llama','gemini','gpt4']
        },
    color_discrete_map=COLOURS,
    template='simple_white'
)
fig.show()


,mean,sem
condition,,
cognitive_layer,1.316832,0.112515
human_therapist,0.769231,0.261991
standalone_llms,0.910000,0.107398


Shapiro-Wilk: 0.987, p = 0.07
Kolmogorov-Smirnov: 0.065, p = 0.35
[2 x 4 model]
                                        sum_sq     df          F  \
C(model)                              4.982889    3.0   2.331980   
C(condition)                          5.302898    1.0   7.445230   
C(model):C(condition)                 0.760493    3.0   0.355909   
mood_t0                              63.660811    1.0  89.379313   
belief_human                          2.008584    1.0   2.820037   
C(model):belief_human                 2.257884    3.0   1.056684   
C(condition):belief_human             0.418214    1.0   0.587169   
C(model):C(condition):belief_human    3.800109    3.0   1.778442   
feel_human                           18.983219    1.0  26.652300   
C(model):feel_human                   5.349158    3.0   2.503393   
C(condition):feel_human               0.076415    1.0   0.107286   
C(model):C(condition):feel_human      0.729468    3.0   0.341389   
Residual                            

Shapiro-Wilk: 0.981, p = 0.004**
Kolmogorov-Smirnov: 0.056, p = 0.45
[3-way model]
                               sum_sq     df           F        PR(>F)  \
C(condition)                 6.048850    2.0    3.812406  2.358944e-02   
mood_t0                     82.377434    1.0  103.839984  3.529406e-20   
belief_human                 0.494134    1.0    0.622875  4.308423e-01   
C(condition):belief_human    6.156408    2.0    3.880197  2.209452e-02   
feel_human                  29.054908    1.0   36.624850  6.214950e-09   
C(condition):feel_human      0.415848    2.0    0.262096  7.696802e-01   
Residual                   172.148555  217.0         NaN           NaN   

                                     p  partial_n2  
C(condition)                    0.024*    0.033945  
mood_t0                    3.53e-20***    0.323650  
belief_human                      0.43    0.002862  
C(condition):belief_human       0.022*    0.034527  
feel_human                 6.21e-09***    0.144406  
C(cond

### Controlling for latency & session duration

In [9]:
df = df_users[['pid','mood_t0','mood_t1','feel_human','belief_human']].copy()
df['mood_change'] = df['mood_t1'] - df['mood_t0']

df = df.merge(df_pids[['pid','condition','model','session_duration_minutes','mean_response_latency_seconds']],on='pid',how='left')

# ------------------------------------------------------------------------------------------------------------------------------------
# Run models
# ------------------------------------------------------------------------------------------------------------------------------------

# --- 2 x 4 model
model_df = df.loc[df['condition']!='human_therapist',].copy().reset_index(drop=True)

model_fit = smf.ols(
    f'mood_change ~ mood_t0 + C(model) * C(condition) * belief_human + C(model) * C(condition) * feel_human + session_duration_minutes + mean_response_latency_seconds',
    data=model_df).fit()

display_residuals(model_fit)
anova_table = format_anova_table(model_fit)

print('[2 x 4 model]')
print(anova_table)

# --- 3-way model
model_df = df.groupby(['pid','condition'])[[
    f'mood_change','mood_t0','belief_human','feel_human','session_duration_minutes','mean_response_latency_seconds'
    ]].mean().reset_index()

model_fit = smf.ols(
    f'mood_change ~ mood_t0 + C(condition)*belief_human + C(condition)*feel_human + session_duration_minutes + mean_response_latency_seconds',
    data=model_df).fit()

display_residuals(model_fit)
anova_table = format_anova_table(model_fit)

print('[3-way model]')
print(anova_table)

print(' ')
print(anova_pairwise_comparisons(model_fit))

Shapiro-Wilk: 0.987, p = 0.06
Kolmogorov-Smirnov: 0.075, p = 0.20
[2 x 4 model]
                                        sum_sq     df          F  \
C(model)                              1.066898    3.0   0.497280   
C(condition)                          0.211382    1.0   0.295574   
C(model):C(condition)                 0.667886    3.0   0.311301   
mood_t0                              63.933319    1.0  89.397682   
belief_human                          1.407691    1.0   1.968368   
C(model):belief_human                 2.182848    3.0   1.017422   
C(condition):belief_human             0.246497    1.0   0.344675   
C(model):C(condition):belief_human    3.492518    3.0   1.627858   
feel_human                           18.472230    1.0  25.829639   
C(model):feel_human                   4.902856    3.0   2.285214   
C(condition):feel_human               0.023368    1.0   0.032676   
C(model):C(condition):feel_human      0.703859    3.0   0.328068   
session_duration_minutes            

Shapiro-Wilk: 0.980, p = 0.003**
Kolmogorov-Smirnov: 0.055, p = 0.47
[3-way model]
                                   sum_sq     df           F        PR(>F)  \
C(condition)                     2.862472    2.0    1.803506  1.672033e-01   
mood_t0                         81.731611    1.0  102.990310  5.053395e-20   
belief_human                     0.281766    1.0    0.355055  5.518925e-01   
C(condition):belief_human        5.602803    2.0    3.530057  3.101341e-02   
feel_human                      27.370358    1.0   34.489491  1.611470e-08   
C(condition):feel_human          0.403119    2.0    0.253986  7.759353e-01   
session_duration_minutes         1.455593    1.0    1.834198  1.770541e-01   
mean_response_latency_seconds    0.001605    1.0    0.002022  9.641726e-01   
Residual                       170.620869  215.0         NaN           NaN   

                                         p  partial_n2  
C(condition)                          0.17    0.016500  
mood_t0               